In [1]:
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

transactions = pd.read_csv("../data/raw/Transactions.csv")
products = pd.read_csv("../data/raw/Products.csv")

print("Transactions shape:", transactions.shape)
print("Products shape:", products.shape)

Transactions shape: (53820, 11)
Products shape: (96, 6)


In [3]:
print("TRANSACTIONS COLUMNS")
print(transactions.columns.tolist())

print("\nPRODUCTS COLUMNS")
print(products.columns.tolist())

TRANSACTIONS COLUMNS
['Outlet', 'Date', 'Time', 'NetSales', 'Tax', 'TotalAmount', 'ReceiptNumber', 'Items', 'TotalItem', 'PaymentMethod', 'UseLoyaltyCard']

PRODUCTS COLUMNS
['ProductId', 'ProductName', 'Variant', 'Category', 'Price', 'Description']


In [4]:
transactions.head()

,Outlet,Date,Time,NetSales,Tax,TotalAmount,ReceiptNumber,Items,TotalItem,PaymentMethod,UseLoyaltyCard
0,SHOP001,2025-01-02,07:01:05,20720.721,2279.279,23000,SHOP001JAN2500001,Basic Latte (Ice Arabica),1,BCA,0
1,SHOP001,2025-01-02,07:04:11,18018.018,1981.982,20000,SHOP001JAN2500002,BLACK (Houseblend),1,BCA,0
2,SHOP002,2025-01-02,07:10:46,39639.640,4360.360,44000,SHOP002JAN2500001,"Friendly Coffee (Hot), Shakencano",2,Cash,0
3,SHOP002,2025-01-02,07:14:25,36036.036,3963.964,40000,SHOP002JAN2500002,"BLACK (Arabica), Americano (Hot Arabica)",2,Cash,0
4,SHOP002,2025-01-02,07:23:05,36036.036,3963.964,40000,SHOP002JAN2500003,"WHITE (Arabica), Susu Oat",2,Cash,0


In [5]:
products.head()

,ProductId,ProductName,Variant,Category,Price,Description
0,1,Add on Aren,-,Add on,0,"Tambahan sirup gula aren murni, memberikan ras..."
1,2,Add on Simple Syrup,-,Add on,0,Tambahan sirup gula sederhana (air dan gula) s...
2,3,Air Mineral,-,Other,5000,Air minum murni dalam kemasan
3,4,Americano,Hot Houseblend,Coffee,20000,Minuman kopi hitam yang dibuat dengan menambah...
4,5,Americano,Ice Houesblend,Coffee,20000,Minuman kopi hitam yang dibuat dengan menambah...


In [6]:
print("TRANSACTIONS INFO")
transactions.info()

print("\nPRODUCTS INFO")
products.info()

TRANSACTIONS INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53820 entries, 0 to 53819
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Outlet          53820 non-null  object 
 1   Date            53820 non-null  object 
 2   Time            53820 non-null  object 
 3   NetSales        53820 non-null  float64
 4   Tax             53820 non-null  float64
 5   TotalAmount     53820 non-null  int64  
 6   ReceiptNumber   53820 non-null  object 
 7   Items           53820 non-null  object 
 8   TotalItem       53820 non-null  int64  
 9   PaymentMethod   53820 non-null  object 
 10  UseLoyaltyCard  53820 non-null  int64  
dtypes: float64(2), int64(3), object(6)
memory usage: 4.5+ MB

PRODUCTS INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ProductId    96 non-nu

In [7]:
print("=== DATE RANGE ===")
print("Start:", transactions["Date"].min())
print("End:", transactions["Date"].max())

print("\n=== OUTLETS ===")
print(transactions["Outlet"].unique())
print("Total outlet:", transactions["Outlet"].nunique())

print("\n=== TOTAL ITEM DISTRIBUTION ===")
print(transactions["TotalItem"].value_counts().sort_index())

=== DATE RANGE ===
Start: 2025-01-02
End: 2025-09-30

=== OUTLETS ===
['SHOP001' 'SHOP002' 'SHOP003']
Total outlet: 3

=== TOTAL ITEM DISTRIBUTION ===
TotalItem
1    45667
2     8153
Name: count, dtype: int64


In [8]:
print("=== SAMPLE ITEMS ===")

for item in transactions["Items"].sample(20, random_state=42):
    print(item)

=== SAMPLE ITEMS ===
Jus Semangka
Friendly Coffee (Gedhe)
Lemons
Matcha Latte (Hot)
Vanilla Latte (Gedhe)
Americano (Hot Houseblend)
Chamomile
Matcha Latte (Ice)
Berry Cake
Candy Latte
Pisang Coklat
Not Tiramisu Latte (Gedhe)
Friendly Coffee (Hot)
Creamy Chocolate Cake
Green Thai Tea
Apple Pie Latte (Gedhe)
Air Mineral
Biru
Burnt Sugar
Susu Oat, Thai Tea (Gedhe)


In [9]:
print("Duplicate rows:", transactions.duplicated().sum())

print(
    "Duplicate ReceiptNumber:",
    transactions["ReceiptNumber"].duplicated().sum()
)

Duplicate rows: 0
Duplicate ReceiptNumber: 0


In [10]:
# Hitung jumlah item berdasarkan pemisah koma
transactions["ParsedItemCount"] = (
    transactions["Items"]
    .str.split(",")
    .str.len()
)

# Bandingkan dengan TotalItem
count_check = (
    transactions["ParsedItemCount"]
    == transactions["TotalItem"]
)

print("Jumlah transaksi:", len(transactions))
print("Jumlah cocok:", count_check.sum())
print("Jumlah tidak cocok:", (~count_check).sum())
print("Persentase cocok:", count_check.mean() * 100, "%")

Jumlah transaksi: 53820
Jumlah cocok: 53820
Jumlah tidak cocok: 0
Persentase cocok: 100.0 %


In [11]:
transactions.loc[
    transactions["ParsedItemCount"] != transactions["TotalItem"],
    ["ReceiptNumber", "Items", "TotalItem", "ParsedItemCount"]
].head(20)

,ReceiptNumber,Items,TotalItem,ParsedItemCount


In [12]:
products["ItemName"] = products.apply(
    lambda row: (
        row["ProductName"]
        if row["Variant"] == "-"
        else f'{row["ProductName"]} ({row["Variant"]})'
    ),
    axis=1
)

products[["ProductName", "Variant", "ItemName"]].head(20)

,ProductName,Variant,ItemName
0,Add on Aren,-,Add on Aren
1,Add on Simple Syrup,-,Add on Simple Syrup
2,Air Mineral,-,Air Mineral
3,Americano,Hot Houseblend,Americano (Hot Houseblend)
4,Americano,Ice Houesblend,Americano (Ice Houesblend)
5,Americano,Hot Arabica,Americano (Hot Arabica)
6,Americano,Ice Arabica,Americano (Ice Arabica)
7,Apple Pie Latte,-,Apple Pie Latte
8,Apple Pie Latte (Gedhe),-,Apple Pie Latte (Gedhe)
9,BLACK,-,BLACK


In [13]:
transaction_items = (
    transactions["Items"]
    .str.split(",")
    .explode()
    .str.strip()
)

print("Jumlah item setelah explode:", len(transaction_items))
print("Jumlah nama item unik:", transaction_items.nunique())

transaction_items.unique()[:30]

Jumlah item setelah explode: 61973
Jumlah nama item unik: 94


array(['Basic Latte (Ice Arabica)', 'BLACK (Houseblend)',
       'Friendly Coffee (Hot)', 'Shakencano', 'BLACK (Arabica)',
       'Americano (Hot Arabica)', 'WHITE (Arabica)', 'Susu Oat',
       'Americano (Hot Houseblend)', 'Banana Bread (Chocolate)',
       'Not Tiramisu Latte (Gedhe)', 'Espresso Arabica (On The Rock)',
       'Jasmine (Hot)', 'Jus Semangka', 'Sitrus Cafe',
       'Hazelnut Latte (Gedhe)', 'Gula Gula', 'Happy Moca (Ice)',
       'Friendly Coffee (Ice)', 'Vanilla Latte (Gedhe)', 'Sunset',
       'Not Tiramisu Latte', 'Green Thai Tea',
       'Cappuccino (Ice Houseblend)', 'Espresso Arabica (Basic Espresso)',
       'Burnt Sugar', 'Espresso Houseblend (On The Rock)',
       'Basic Latte (Hot Arabica)', 'Friendly Coffee (Hot double shot)',
       'Chocolate (Ice)'], dtype=object)

In [14]:
product_names = set(products["ItemName"])

unmatched_items = sorted(
    set(transaction_items) - product_names
)

print("Jumlah unmatched item:", len(unmatched_items))

unmatched_items[:50]

Jumlah unmatched item: 0


[]

In [15]:
# Buat copy agar data asli tidak berubah
items_df = transactions.copy()

# Pecah Items berdasarkan koma
items_df["ItemName"] = items_df["Items"].str.split(",")

# Ubah setiap item menjadi satu baris
items_df = items_df.explode("ItemName")

# Bersihkan spasi
items_df["ItemName"] = items_df["ItemName"].str.strip()

print("Transactions awal :", len(transactions))
print("Setelah explode   :", len(items_df))

items_df.head(10)

Transactions awal : 53820
Setelah explode   : 61973


,Outlet,Date,Time,NetSales,Tax,TotalAmount,ReceiptNumber,Items,TotalItem,PaymentMethod,UseLoyaltyCard,ParsedItemCount,ItemName
0,SHOP001,2025-01-02,07:01:05,20720.721,2279.279,23000,SHOP001JAN2500001,Basic Latte (Ice Arabica),1,BCA,0,1,Basic Latte (Ice Arabica)
1,SHOP001,2025-01-02,07:04:11,18018.018,1981.982,20000,SHOP001JAN2500002,BLACK (Houseblend),1,BCA,0,1,BLACK (Houseblend)
2,SHOP002,2025-01-02,07:10:46,39639.640,4360.360,44000,SHOP002JAN2500001,"Friendly Coffee (Hot), Shakencano",2,Cash,0,2,Friendly Coffee (Hot)
2,SHOP002,2025-01-02,07:10:46,39639.640,4360.360,44000,SHOP002JAN2500001,"Friendly Coffee (Hot), Shakencano",2,Cash,0,2,Shakencano
3,SHOP002,2025-01-02,07:14:25,36036.036,3963.964,40000,SHOP002JAN2500002,"BLACK (Arabica), Americano (Hot Arabica)",2,Cash,0,2,BLACK (Arabica)
3,SHOP002,2025-01-02,07:14:25,36036.036,3963.964,40000,SHOP002JAN2500002,"BLACK (Arabica), Americano (Hot Arabica)",2,Cash,0,2,Americano (Hot Arabica)
4,SHOP002,2025-01-02,07:23:05,36036.036,3963.964,40000,SHOP002JAN2500003,"WHITE (Arabica), Susu Oat",2,Cash,0,2,WHITE (Arabica)
4,SHOP002,2025-01-02,07:23:05,36036.036,3963.964,40000,SHOP002JAN2500003,"WHITE (Arabica), Susu Oat",2,Cash,0,2,Susu Oat
5,SHOP001,2025-01-02,07:40:48,18018.018,1981.982,20000,SHOP001JAN2500003,Americano (Hot Houseblend),1,BCA,0,1,Americano (Hot Houseblend)
6,SHOP002,2025-01-02,07:42:40,10810.811,1189.189,12000,SHOP002JAN2500004,Banana Bread (Chocolate),1,Cash,0,1,Banana Bread (Chocolate)


In [16]:
sample_receipt = transactions.loc[
    transactions["TotalItem"] == 2,
    "ReceiptNumber"
].iloc[0]

print("Receipt:", sample_receipt)

items_df.loc[
    items_df["ReceiptNumber"] == sample_receipt,
    [
        "Date",
        "Outlet",
        "ReceiptNumber",
        "ItemName",
        "TotalItem"
    ]
]

Receipt: SHOP002JAN2500001


,Date,Outlet,ReceiptNumber,ItemName,TotalItem
2,2025-01-02,SHOP002,SHOP002JAN2500001,Friendly Coffee (Hot),2
2,2025-01-02,SHOP002,SHOP002JAN2500001,Shakencano,2


In [17]:
print("Jumlah baris:", len(items_df))
print("Missing ItemName:", items_df["ItemName"].isna().sum())
print("Item kosong:", (items_df["ItemName"] == "").sum())
print("Receipt unik:", items_df["ReceiptNumber"].nunique())

Jumlah baris: 61973
Missing ItemName: 0
Item kosong: 0
Receipt unik: 53820


In [18]:
items_merged = items_df.merge(
    products[
        [
            "ProductId",
            "ProductName",
            "Variant",
            "Category",
            "Price",
            "ItemName"
        ]
    ],
    on="ItemName",
    how="left",
    validate="many_to_one"
)

print("Sebelum merge :", len(items_df))
print("Setelah merge :", len(items_merged))

items_merged.head()

Sebelum merge : 61973
Setelah merge : 61973


,Outlet,Date,Time,NetSales,Tax,TotalAmount,ReceiptNumber,Items,TotalItem,PaymentMethod,UseLoyaltyCard,ParsedItemCount,ItemName,ProductId,ProductName,Variant,Category,Price
0,SHOP001,2025-01-02,07:01:05,20720.721,2279.279,23000,SHOP001JAN2500001,Basic Latte (Ice Arabica),1,BCA,0,1,Basic Latte (Ice Arabica),18,Basic Latte,Ice Arabica,Coffee,23000
1,SHOP001,2025-01-02,07:04:11,18018.018,1981.982,20000,SHOP001JAN2500002,BLACK (Houseblend),1,BCA,0,1,BLACK (Houseblend),11,BLACK,Houseblend,Menu Gedhe,20000
2,SHOP002,2025-01-02,07:10:46,39639.640,4360.360,44000,SHOP002JAN2500001,"Friendly Coffee (Hot), Shakencano",2,Cash,0,2,Friendly Coffee (Hot),49,Friendly Coffee,Hot,Coffee,23000
3,SHOP002,2025-01-02,07:10:46,39639.640,4360.360,44000,SHOP002JAN2500001,"Friendly Coffee (Hot), Shakencano",2,Cash,0,2,Shakencano,79,Shakencano,-,Coffee,21000
4,SHOP002,2025-01-02,07:14:25,36036.036,3963.964,40000,SHOP002JAN2500002,"BLACK (Arabica), Americano (Hot Arabica)",2,Cash,0,2,BLACK (Arabica),12,BLACK,Arabica,Menu Gedhe,20000


In [19]:
print("=== MISSING SETELAH MERGE ===")

print(
    items_merged[
        ["ProductId", "ProductName", "Variant", "Category", "Price"]
    ]
    .isna()
    .sum()
)

=== MISSING SETELAH MERGE ===
ProductId      0
ProductName    0
Variant        0
Category       0
Price          0
dtype: int64


In [20]:
items_merged["Quantity"] = 1

In [21]:
items_merged[
    [
        "Date",
        "Outlet",
        "ProductId",
        "ProductName",
        "Variant",
        "Category",
        "Quantity"
    ]
].head(10)

,Date,Outlet,ProductId,ProductName,Variant,Category,Quantity
0,2025-01-02,SHOP001,18,Basic Latte,Ice Arabica,Coffee,1
1,2025-01-02,SHOP001,11,BLACK,Houseblend,Menu Gedhe,1
2,2025-01-02,SHOP002,49,Friendly Coffee,Hot,Coffee,1
3,2025-01-02,SHOP002,79,Shakencano,-,Coffee,1
4,2025-01-02,SHOP002,12,BLACK,Arabica,Menu Gedhe,1
5,2025-01-02,SHOP002,6,Americano,Hot Arabica,Coffee,1
6,2025-01-02,SHOP002,92,WHITE,Arabica,Menu Gedhe,1
7,2025-01-02,SHOP002,84,Susu Oat,-,Add on,1
8,2025-01-02,SHOP001,4,Americano,Hot Houseblend,Coffee,1
9,2025-01-02,SHOP002,13,Banana Bread,Chocolate,Snacks,1


In [22]:
items_merged["Date"] = pd.to_datetime(items_merged["Date"])

print(items_merged["Date"].dtype)

datetime64[ns]


In [23]:
daily_sales = (
    items_merged
    .groupby(
        [
            "Date",
            "Outlet",
            "ProductId",
            "ProductName",
            "Variant",
            "Category"
        ],
        as_index=False
    )
    .agg(
        QuantitySold=("Quantity", "sum")
    )
)

In [24]:
daily_sales.head(20)

,Date,Outlet,ProductId,ProductName,Variant,Category,QuantitySold
0,2025-01-02,SHOP001,3,Air Mineral,-,Other,2
1,2025-01-02,SHOP001,4,Americano,Hot Houseblend,Coffee,1
2,2025-01-02,SHOP001,5,Americano,Ice Houesblend,Coffee,1
3,2025-01-02,SHOP001,7,Americano,Ice Arabica,Coffee,2
4,2025-01-02,SHOP001,9,Apple Pie Latte (Gedhe),-,Menu Gedhe,1
5,2025-01-02,SHOP001,10,BLACK,-,Menu Gedhe,1
6,2025-01-02,SHOP001,11,BLACK,Houseblend,Menu Gedhe,1
7,2025-01-02,SHOP001,12,BLACK,Arabica,Menu Gedhe,2
8,2025-01-02,SHOP001,13,Banana Bread,Chocolate,Snacks,1
9,2025-01-02,SHOP001,15,Basic Latte,Hot Arabica,Coffee,1


In [25]:
print("Jumlah baris daily sales:", len(daily_sales))

print(
    "Total quantity setelah agregasi:",
    daily_sales["QuantitySold"].sum()
)

print(
    "Total item sebelum agregasi:",
    len(items_merged)
)

Jumlah baris daily sales: 40789
Total quantity setelah agregasi: 61973
Total item sebelum agregasi: 61973


In [26]:
print("Missing values:")
print(daily_sales.isna().sum())

print("\nDuplicate combination:")
print(
    daily_sales.duplicated(
        subset=["Date", "Outlet", "ProductId"]
    ).sum()
)

Missing values:
Date            0
Outlet          0
ProductId       0
ProductName     0
Variant         0
Category        0
QuantitySold    0
dtype: int64

Duplicate combination:
0


In [27]:
date_check = (
    transactions
    .groupby("Date")
    .size()
    .reset_index(name="TransactionCount")
)

print("Jumlah tanggal yang punya transaksi:", len(date_check))

print("Tanggal awal:", transactions["Date"].min())
print("Tanggal akhir:", transactions["Date"].max())

Jumlah tanggal yang punya transaksi: 272
Tanggal awal: 2025-01-02
Tanggal akhir: 2025-09-30


In [28]:
# ============================================================
# CORE SKU SELECTION — COFFEE PRODUCTS
# ============================================================

coffee_sales = daily_sales[
    daily_sales["Category"].str.lower() == "coffee"
].copy()

print("Coffee ProductId:", coffee_sales["ProductId"].nunique())
print("Coffee ProductName:", coffee_sales["ProductName"].nunique())
print("Outlet:", coffee_sales["Outlet"].nunique())

coffee_sales.head()

Coffee ProductId: 34
Coffee ProductName: 19
Outlet: 3


,Date,Outlet,ProductId,ProductName,Variant,Category,QuantitySold
1,2025-01-02,SHOP001,4,Americano,Hot Houseblend,Coffee,1
2,2025-01-02,SHOP001,5,Americano,Ice Houesblend,Coffee,1
3,2025-01-02,SHOP001,7,Americano,Ice Arabica,Coffee,2
9,2025-01-02,SHOP001,15,Basic Latte,Hot Arabica,Coffee,1
10,2025-01-02,SHOP001,16,Basic Latte,Ice Houseblend,Coffee,2


In [29]:
start_date = daily_sales["Date"].min()
end_date = daily_sales["Date"].max()

all_dates = pd.date_range(
    start=start_date,
    end=end_date,
    freq="D"
)

total_days = len(all_dates)

print("Start date :", start_date)
print("End date   :", end_date)
print("Total days :", total_days)

Start date : 2025-01-02 00:00:00
End date   : 2025-09-30 00:00:00
Total days : 272


In [ ]:
# Ranking berdasarkan volume + sparsity

coffee_ranking = (
    coffee_sales
    .groupby(
        [
            "ProductId",
            "ProductName",
            "Variant"
        ],
        as_index=False
    )
    .agg(
        TotalQtySold=("QuantitySold", "sum"),
        ActiveDays=("Date", "nunique"),
        OutletCoverage=("Outlet", "nunique"),
        AvgQtyOnSellingDay=("QuantitySold", "mean")
    )
)

# Berapa persen hari produk memiliki penjualan
coffee_ranking["SalesCoverage"] = (
    coffee_ranking["ActiveDays"] / total_days
)

# Berapa persen hari tidak memiliki penjualan
coffee_ranking["Sparsity"] = (
    1 - coffee_ranking["SalesCoverage"]
)

coffee_ranking.head()

,ProductId,ProductName,Variant,TotalQtySold,ActiveDays,OutletCoverage,AvgQtyOnSellingDay,SalesCoverage,Sparsity
0,4,Americano,Hot Houseblend,641,242,3,1.487239,0.889706,0.110294
1,5,Americano,Ice Houesblend,686,251,3,1.527840,0.922794,0.077206
2,6,Americano,Hot Arabica,706,244,3,1.568889,0.897059,0.102941
3,7,Americano,Ice Arabica,633,242,3,1.578554,0.889706,0.110294
4,8,Apple Pie Latte,-,670,250,3,1.536697,0.919118,0.080882


In [ ]:
#Urutkan berdasarkan volume tertinggi, kemudian sparsity terendah:

coffee_ranking = (
    coffee_ranking
    .sort_values(
        by=[
            "TotalQtySold",
            "Sparsity",
            "OutletCoverage"
        ],
        ascending=[
            False,
            True,
            False
        ]
    )
    .reset_index(drop=True)
)

coffee_ranking["Rank"] = (
    coffee_ranking.index + 1
)

coffee_ranking = coffee_ranking[
    [
        "Rank",
        "ProductId",
        "ProductName",
        "Variant",
        "TotalQtySold",
        "ActiveDays",
        "SalesCoverage",
        "Sparsity",
        "OutletCoverage",
        "AvgQtyOnSellingDay"
    ]
]

coffee_ranking.head(34)

,Rank,ProductId,ProductName,Variant,TotalQtySold,ActiveDays,SalesCoverage,Sparsity,OutletCoverage,AvgQtyOnSellingDay
0,1,50,Friendly Coffee,Ice,715,253,0.930147,0.069853,3,1.550976
1,2,49,Friendly Coffee,Hot,708,241,0.886029,0.113971,3,1.573333
2,3,6,Americano,Hot Arabica,706,244,0.897059,0.102941,3,1.568889
3,4,25,Butterscotch,-,701,250,0.919118,0.080882,3,1.550885
4,5,79,Shakencano,-,699,251,0.922794,0.077206,3,1.546460
5,6,5,Americano,Ice Houesblend,686,251,0.922794,0.077206,3,1.527840
6,7,56,Happy Moca,Hot,680,252,0.926471,0.073529,3,1.528090
7,8,80,Sitrus Cafe,-,674,245,0.900735,0.099265,3,1.507830
8,9,88,Vanilla Latte,-,673,247,0.908088,0.091912,3,1.502232
9,10,66,Matcha Espresso,-,671,250,0.919118,0.080882,3,1.535469


In [ ]:
# Outlet × ProductId × Date

coffee_outlet_stats = (
    coffee_sales
    .groupby(
        [
            "ProductId",
            "ProductName",
            "Variant",
            "Outlet"
        ],
        as_index=False
    )
    .agg(
        TotalQtySold=("QuantitySold", "sum"),
        ActiveDays=("Date", "nunique")
    )
)

coffee_outlet_stats["SalesCoverage"] = (
    coffee_outlet_stats["ActiveDays"] / total_days
)

coffee_outlet_stats["Sparsity"] = (
    1 - coffee_outlet_stats["SalesCoverage"]
)

coffee_outlet_stats["SalesCoveragePct"] = (
    coffee_outlet_stats["SalesCoverage"] * 100
).round(2)

coffee_outlet_stats["SparsityPct"] = (
    coffee_outlet_stats["Sparsity"] * 100
).round(2)

coffee_outlet_stats.sort_values(
    ["ProductId", "Outlet"]
).head(30)

,ProductId,ProductName,Variant,Outlet,TotalQtySold,ActiveDays,SalesCoverage,Sparsity,SalesCoveragePct,SparsityPct
0,4,Americano,Hot Houseblend,SHOP001,282,176,0.647059,0.352941,64.71,35.29
1,4,Americano,Hot Houseblend,SHOP002,142,107,0.393382,0.606618,39.34,60.66
2,4,Americano,Hot Houseblend,SHOP003,217,148,0.544118,0.455882,54.41,45.59
3,5,Americano,Ice Houesblend,SHOP001,329,197,0.724265,0.275735,72.43,27.57
4,5,Americano,Ice Houesblend,SHOP002,145,106,0.389706,0.610294,38.97,61.03
5,5,Americano,Ice Houesblend,SHOP003,212,146,0.536765,0.463235,53.68,46.32
6,6,Americano,Hot Arabica,SHOP001,340,189,0.694853,0.305147,69.49,30.51
7,6,Americano,Hot Arabica,SHOP002,153,116,0.426471,0.573529,42.65,57.35
8,6,Americano,Hot Arabica,SHOP003,213,145,0.533088,0.466912,53.31,46.69
9,7,Americano,Ice Arabica,SHOP001,318,180,0.661765,0.338235,66.18,33.82


In [ ]:
# matriks sparsity SKU × outlet

sparsity_matrix = (
    coffee_outlet_stats
    .pivot_table(
        index=[
            "ProductId",
            "ProductName",
            "Variant"
        ],
        columns="Outlet",
        values="SparsityPct"
    )
    .reset_index()
)

sparsity_matrix.head(34)

Outlet,ProductId,ProductName,Variant,SHOP001,SHOP002,SHOP003
0,4,Americano,Hot Houseblend,35.29,60.66,45.59
1,5,Americano,Ice Houesblend,27.57,61.03,46.32
2,6,Americano,Hot Arabica,30.51,57.35,46.69
3,7,Americano,Ice Arabica,33.82,66.54,52.21
4,8,Apple Pie Latte,-,35.66,61.40,42.65
5,15,Basic Latte,Hot Arabica,31.25,60.66,48.53
6,16,Basic Latte,Ice Houseblend,29.41,66.54,47.06
7,17,Basic Latte,Hot Houseblend,31.25,64.34,51.84
8,18,Basic Latte,Ice Arabica,24.26,60.29,53.68
9,20,Biru,-,34.56,61.76,50.00


CORE SKU SELECTION — 15 COFFEE PRODUCTS
Criteria:
1. TotalQtySold      -> business importance
2. ActiveDays        -> demand continuity
3. OutletCoverage    -> multi-outlet relevance
4. GlobalSparsity    -> overall intermittency
5. OutletSparsity    -> forecastability per outlet

In [38]:
# 1. Filter Coffee products

coffee_sales = daily_sales[
    daily_sales["Category"].str.lower().eq("coffee")
].copy()

# Memastikan Date sudah datetime
coffee_sales["Date"] = pd.to_datetime(coffee_sales["Date"])

# Gunakan periode observasi yang sama untuk semua SKU
start_date = daily_sales["Date"].min()
end_date = daily_sales["Date"].max()

total_days = (
    pd.to_datetime(end_date) - pd.to_datetime(start_date)
).days + 1

print(f"Observation period : {start_date} -> {end_date}")
print(f"Total days         : {total_days}")
print(f"Coffee SKU         : {coffee_sales['ProductId'].nunique()}")

Observation period : 2025-01-02 00:00:00 -> 2025-09-30 00:00:00
Total days         : 272
Coffee SKU         : 34


In [39]:
# 2. Global statistics per ProductId

global_stats = (
    coffee_sales
    .groupby(
        ["ProductId", "ProductName", "Variant"],
        as_index=False
    )
    .agg(
        TotalQtySold=("QuantitySold", "sum"),
        ActiveDays=("Date", "nunique"),
        OutletCoverage=("Outlet", "nunique")
    )
)

global_stats["SalesCoverage"] = (
    global_stats["ActiveDays"] / total_days
)

global_stats["GlobalSparsity"] = (
    1 - global_stats["SalesCoverage"]
)

In [40]:
# 3. Statistics per ProductId × Outlet
outlet_stats = (
    coffee_sales
    .groupby(
        [
            "ProductId",
            "ProductName",
            "Variant",
            "Outlet"
        ],
        as_index=False
    )
    .agg(
        OutletQtySold=("QuantitySold", "sum"),
        OutletActiveDays=("Date", "nunique")
    )
)

outlet_stats["OutletSalesCoverage"] = (
    outlet_stats["OutletActiveDays"] / total_days
)

outlet_stats["OutletSparsity"] = (
    1 - outlet_stats["OutletSalesCoverage"]
)

In [43]:
# 4. Summarize outlet-level intermittency

outlet_summary = (
    outlet_stats
    .groupby("ProductId", as_index=False)
    .agg(
        AvgOutletSparsity=("OutletSparsity", "mean"),
        MaxOutletSparsity=("OutletSparsity", "max"),
        MinOutletSparsity=("OutletSparsity", "min"),
        MinOutletActiveDays=("OutletActiveDays", "min")
    )
)

In [44]:
# 5. Combine all selection criteria

sku_stats = global_stats.merge(
    outlet_summary,
    on="ProductId",
    how="left"
)

sku_stats.head()

,ProductId,ProductName,Variant,TotalQtySold,ActiveDays,OutletCoverage,SalesCoverage,GlobalSparsity,AvgOutletSparsity,MaxOutletSparsity,MinOutletSparsity,MinOutletActiveDays
0,4,Americano,Hot Houseblend,641,242,3,0.889706,0.110294,0.471814,0.606618,0.352941,107
1,5,Americano,Ice Houesblend,686,251,3,0.922794,0.077206,0.449755,0.610294,0.275735,106
2,6,Americano,Hot Arabica,706,244,3,0.897059,0.102941,0.448529,0.573529,0.305147,116
3,7,Americano,Ice Arabica,633,242,3,0.889706,0.110294,0.508578,0.665441,0.338235,91
4,8,Apple Pie Latte,-,670,250,3,0.919118,0.080882,0.465686,0.613971,0.356618,105


In [45]:
# 6. Eligibility filter

MAX_GLOBAL_SPARSITY = 0.15
MAX_OUTLET_SPARSITY = 0.625
REQUIRED_OUTLETS = 3

eligible_skus = sku_stats[
    (sku_stats["OutletCoverage"] == REQUIRED_OUTLETS) &
    (sku_stats["GlobalSparsity"] <= MAX_GLOBAL_SPARSITY) &
    (sku_stats["MaxOutletSparsity"] <= MAX_OUTLET_SPARSITY)
].copy()

print("Eligible SKU:", len(eligible_skus))

Eligible SKU: 23


In [47]:
# 7. Rank eligible SKU

eligible_skus = (
    eligible_skus
    .sort_values(
        by=[
            "TotalQtySold",        # higher = better
            "ActiveDays",          # higher = better
            "MaxOutletSparsity",   # lower = better
            "AvgOutletSparsity",   # lower = better
            "GlobalSparsity"       # lower = better
        ],
        ascending=[
            False,
            False,
            True,
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

eligible_skus["Rank"] = eligible_skus.index + 1

In [48]:
# 8. Select final 15 Core SKU

core_15 = eligible_skus.head(15).copy()

core_15 = core_15[
    [
        "Rank",
        "ProductId",
        "ProductName",
        "Variant",
        "TotalQtySold",
        "ActiveDays",
        "OutletCoverage",
        "GlobalSparsity",
        "AvgOutletSparsity",
        "MaxOutletSparsity"
    ]
]

# Versi display persen
core_15_display = core_15.copy()

for col in [
    "GlobalSparsity",
    "AvgOutletSparsity",
    "MaxOutletSparsity"
]:
    core_15_display[col] = (
        core_15_display[col] * 100
    ).round(2)

display(core_15_display)

,Rank,ProductId,ProductName,Variant,TotalQtySold,ActiveDays,OutletCoverage,GlobalSparsity,AvgOutletSparsity,MaxOutletSparsity
0,1,50,Friendly Coffee,Ice,715,253,3,6.99,43.50,59.19
1,2,49,Friendly Coffee,Hot,708,241,3,11.40,44.85,59.56
2,3,6,Americano,Hot Arabica,706,244,3,10.29,44.85,57.35
3,4,25,Butterscotch,-,701,250,3,8.09,44.61,62.13
4,5,79,Shakencano,-,699,251,3,7.72,44.61,56.25
5,6,5,Americano,Ice Houesblend,686,251,3,7.72,44.98,61.03
6,7,56,Happy Moca,Hot,680,252,3,7.35,45.47,60.29
7,8,80,Sitrus Cafe,-,674,245,3,9.93,45.22,59.93
8,9,88,Vanilla Latte,-,673,247,3,9.19,45.10,58.46
9,10,8,Apple Pie Latte,-,670,250,3,8.09,46.57,61.40


In [49]:
CORE_PRODUCT_IDS = core_15["ProductId"].tolist()

print("Final 15 ProductId:")
print(CORE_PRODUCT_IDS)

Final 15 ProductId:
[50, 49, 6, 25, 79, 5, 56, 80, 88, 8, 43, 18, 57, 30, 15]


In [50]:
core_daily_sales = daily_sales[
    daily_sales["ProductId"].isin(CORE_PRODUCT_IDS)
].copy()

print("Final SKU     :", core_daily_sales["ProductId"].nunique())
print("Final outlets :", core_daily_sales["Outlet"].nunique())
print("Rows          :", len(core_daily_sales))

Final SKU     : 15
Final outlets : 3
Rows          : 6682


In [51]:
core_15_reason = core_15.copy()

core_15_reason["SelectionReason"] = (
    "High sales volume; "
    "continuous demand history; "
    "sold across all 3 outlets; "
    "low global sparsity; "
    "acceptable outlet-level sparsity"
)

display(core_15_reason)

,Rank,ProductId,ProductName,Variant,TotalQtySold,ActiveDays,OutletCoverage,GlobalSparsity,AvgOutletSparsity,MaxOutletSparsity,SelectionReason
0,1,50,Friendly Coffee,Ice,715,253,3,0.069853,0.435049,0.591912,High sales volume; continuous demand history; ...
1,2,49,Friendly Coffee,Hot,708,241,3,0.113971,0.448529,0.595588,High sales volume; continuous demand history; ...
2,3,6,Americano,Hot Arabica,706,244,3,0.102941,0.448529,0.573529,High sales volume; continuous demand history; ...
3,4,25,Butterscotch,-,701,250,3,0.080882,0.446078,0.621324,High sales volume; continuous demand history; ...
4,5,79,Shakencano,-,699,251,3,0.077206,0.446078,0.562500,High sales volume; continuous demand history; ...
5,6,5,Americano,Ice Houesblend,686,251,3,0.077206,0.449755,0.610294,High sales volume; continuous demand history; ...
6,7,56,Happy Moca,Hot,680,252,3,0.073529,0.454657,0.602941,High sales volume; continuous demand history; ...
7,8,80,Sitrus Cafe,-,674,245,3,0.099265,0.452206,0.599265,High sales volume; continuous demand history; ...
8,9,88,Vanilla Latte,-,673,247,3,0.091912,0.450980,0.584559,High sales volume; continuous demand history; ...
9,10,8,Apple Pie Latte,-,670,250,3,0.080882,0.465686,0.613971,High sales volume; continuous demand history; ...


In [52]:
# 1. Metadata 15 SKU final
core_15.to_csv(
    "../data/processed/core_15_skus.csv",
    index=False
)

# 2. Data penjualan harian hanya untuk 15 SKU terpilih
core_daily_sales.to_csv(
    "../data/processed/core_daily_sales.csv",
    index=False
)

print("Saved:")
print("../data/processed/core_15_skus.csv")
print("../data/processed/core_daily_sales.csv")

Saved:
../data/processed/core_15_skus.csv
../data/processed/core_daily_sales.csv
